# Section 02: Ingesting and Processing Data

**CareerAlign GCP Professional Data Engineer**

This notebook covers:
1. Apache Beam pipeline with DirectRunner
2. Beam transforms: ParDo, GroupByKey, Combine, windowing
3. Streaming pipeline patterns with windowing and triggers
4. Dataproc/Spark job patterns
5. Cloud Composer DAG examples
6. Pub/Sub message flow simulation

> **Note**: Cells marked with `# REQUIRES GCP` need a GCP project. Local-only cells run anywhere.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rajvermatx/careeralign.com/blob/main/gcp-pde/notebooks/02-ingesting-processing-data.ipynb)

---
## Setup & Installations

In [ ]:
# Install required packages
!pip install -q apache-beam[gcp] pandas numpy google-cloud-pubsub

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta
import json
import warnings
warnings.filterwarnings('ignore')

print("Setup complete!")

---
## 1. Apache Beam Pipeline (Local DirectRunner)

Demonstrates core Beam concepts: Pipeline, PCollection, ParDo, GroupByKey, Combine.

In [ ]:
import apache_beam as beam
from apache_beam.options.pipeline_options import PipelineOptions

# Create sample event data
sample_events = [
    '{"user_id": "u001", "event": "purchase", "amount": 45.99, "ts": "2025-03-01 10:30:00", "region": "US"}',
    '{"user_id": "u002", "event": "browse", "amount": 0.0, "ts": "2025-03-01 11:00:00", "region": "EU"}',
    '{"user_id": "u001", "event": "purchase", "amount": 23.50, "ts": "2025-03-01 14:20:00", "region": "US"}',
    '{"user_id": "u003", "event": "purchase", "amount": 0.0, "ts": "2025-03-01 09:00:00", "region": "US"}',
    '{"user_id": "u002", "event": "purchase", "amount": 89.99, "ts": "2025-03-01 10:15:00", "region": "EU"}',
    '{"user_id": "", "event": "browse", "amount": 12.00, "ts": "2025-03-01 11:30:00", "region": "US"}',
    '{"user_id": "u001", "event": "browse", "amount": 0.0, "ts": "2025-03-01 08:45:00", "region": "US"}',
    '{"user_id": "u003", "event": "purchase", "amount": 150.00, "ts": "2025-03-01 13:00:00", "region": "APAC"}',
    '{"user_id": "u004", "event": "purchase", "amount": -5.00, "ts": "2025-03-01 15:00:00", "region": "US"}',
    '{"user_id": "u002", "event": "purchase", "amount": 34.50, "ts": "2025-03-01 09:30:00", "region": "EU"}',
]

print(f"Sample events: {len(sample_events)} records")
print("Includes: empty user_id (row 6), negative amount (row 9), zero amounts")

In [ ]:
# --- Define DoFn transforms ---

class ParseJSON(beam.DoFn):
    """Parse JSON string into dict with derived fields."""
    def process(self, element):
        record = json.loads(element)
        ts = datetime.strptime(record['ts'], '%Y-%m-%d %H:%M:%S')
        record['hour'] = ts.hour
        record['day_of_week'] = ts.strftime('%A')
        yield record

class ValidateAndRoute(beam.DoFn):
    """Route records to valid or invalid outputs."""
    def process(self, element):
        if not element.get('user_id'):
            yield beam.pvalue.TaggedOutput('invalid', {**element, 'reason': 'missing_user_id'})
        elif element.get('amount', 0) < 0:
            yield beam.pvalue.TaggedOutput('invalid', {**element, 'reason': 'negative_amount'})
        else:
            yield beam.pvalue.TaggedOutput('valid', element)

class EnrichEvent(beam.DoFn):
    """Add derived features."""
    def process(self, element):
        element['is_purchase'] = 1 if element['event'] == 'purchase' else 0
        element['amount_bucket'] = 'high' if element['amount'] > 50 else 'low'
        yield element

print("DoFn transforms defined:")
print("  1. ParseJSON - parse and add time features")
print("  2. ValidateAndRoute - split valid/invalid")
print("  3. EnrichEvent - add derived features")

In [ ]:
# --- Run the pipeline ---

valid_results = []
invalid_results = []

class CollectValid(beam.DoFn):
    def process(self, element):
        valid_results.append(element)
        yield element

class CollectInvalid(beam.DoFn):
    def process(self, element):
        invalid_results.append(element)
        yield element

with beam.Pipeline(options=PipelineOptions()) as p:
    parsed = (
        p
        | 'Create' >> beam.Create(sample_events)
        | 'Parse' >> beam.ParDo(ParseJSON())
    )
    
    tagged = parsed | 'Validate' >> beam.ParDo(ValidateAndRoute()).with_outputs('valid', 'invalid')
    
    tagged.valid | 'Enrich' >> beam.ParDo(EnrichEvent()) | 'CollectValid' >> beam.ParDo(CollectValid())
    tagged.invalid | 'CollectInvalid' >> beam.ParDo(CollectInvalid())

print(f"Pipeline complete!")
print(f"Valid records: {len(valid_results)}")
print(f"Invalid records: {len(invalid_results)}")
print()
print("Invalid records (dead letter):")
for r in invalid_results:
    print(f"  user_id='{r.get('user_id', '')}', reason={r['reason']}")
print()
print("Valid records:")
pd.DataFrame(valid_results)[['user_id', 'event', 'amount', 'region', 'hour', 'is_purchase', 'amount_bucket']]

In [ ]:
# --- GroupByKey and CombinePerKey ---

user_stats = []

class CollectStats(beam.DoFn):
    def process(self, element):
        user_stats.append(element)
        yield element

with beam.Pipeline(options=PipelineOptions()) as p:
    (
        p
        | 'Create' >> beam.Create(sample_events)
        | 'Parse' >> beam.ParDo(ParseJSON())
        | 'Validate' >> beam.ParDo(ValidateAndRoute()).with_outputs('valid', 'invalid')
    ).valid | (
        'ToKV' >> beam.Map(lambda r: (r['user_id'], r['amount']))
        | 'GroupByUser' >> beam.GroupByKey()
        | 'Aggregate' >> beam.Map(lambda kv: {
            'user_id': kv[0],
            'total_spend': round(sum(kv[1]), 2),
            'num_events': len(kv[1]),
            'avg_spend': round(sum(kv[1]) / len(kv[1]), 2)
        })
        | 'Collect' >> beam.ParDo(CollectStats())
    )

print("=== GroupByKey: Per-User Aggregates ===")
pd.DataFrame(user_stats).sort_values('total_spend', ascending=False)

---
## 2. Windowing Patterns

Demonstrate fixed, sliding, and session windows with simulated event-time data.

In [ ]:
# --- Simulate windowed aggregation ---
# Since DirectRunner windowing with event time needs timestamps on elements,
# we'll simulate the concept with pandas

np.random.seed(42)
n_events = 100

events_df = pd.DataFrame({
    'user_id': np.random.choice(['u001', 'u002', 'u003', 'u004'], n_events),
    'event_time': pd.date_range('2025-03-01 00:00', periods=n_events, freq='3min'),
    'amount': np.random.lognormal(3, 0.8, n_events).round(2)
})

# --- Fixed Windows (5-minute tumbling) ---
events_df['fixed_window_5min'] = events_df['event_time'].dt.floor('5min')
fixed_agg = events_df.groupby('fixed_window_5min').agg(
    event_count=('amount', 'count'),
    total_amount=('amount', 'sum'),
    avg_amount=('amount', 'mean')
).round(2).head(10)

print("=== Fixed Windows (5-minute tumbling) ===")
print(f"100 events -> {len(events_df['fixed_window_5min'].unique())} windows")
print(fixed_agg)
print()

# --- Sliding Windows (10-min window, 5-min slide) ---
# Each event belongs to 2 overlapping windows
print("=== Sliding Windows (10-min window, 5-min slide) ===")
print("Each event appears in 2 overlapping windows.")
print("Use case: 10-minute moving average, updated every 5 minutes.")
print()

# --- Session Windows (gap = 10 min) ---
# Group consecutive events per user with gaps < 10 min
def assign_sessions(group, gap_minutes=10):
    group = group.sort_values('event_time')
    sessions = []
    session_id = 0
    prev_time = None
    for _, row in group.iterrows():
        if prev_time and (row['event_time'] - prev_time) > timedelta(minutes=gap_minutes):
            session_id += 1
        sessions.append(session_id)
        prev_time = row['event_time']
    group['session_id'] = sessions
    return group

session_events = events_df.groupby('user_id', group_keys=False).apply(assign_sessions)
session_agg = session_events.groupby(['user_id', 'session_id']).agg(
    start=('event_time', 'min'),
    end=('event_time', 'max'),
    event_count=('amount', 'count'),
    total_amount=('amount', 'sum')
).round(2)

print("=== Session Windows (10-minute gap) ===")
print(session_agg.head(8))

In [ ]:
# --- Beam Windowing Code Examples ---

windowing_code = """
import apache_beam as beam
from apache_beam import window
from apache_beam.transforms.trigger import (
    AfterWatermark, AfterProcessingTime, AfterCount,
    AccumulationMode
)
from apache_beam.utils.timestamp import Duration

# Fixed windows: 5-minute tumbling
events | beam.WindowInto(window.FixedWindows(5 * 60))

# Sliding windows: 10-min window, slide every 2 min
events | beam.WindowInto(window.SlidingWindows(10 * 60, 2 * 60))

# Session windows: 30-minute gap
events | beam.WindowInto(window.Sessions(30 * 60))

# Advanced: Fixed window with late data handling
events | beam.WindowInto(
    window.FixedWindows(5 * 60),
    trigger=AfterWatermark(
        early=AfterProcessingTime(60),      # Speculative every 60s
        late=AfterCount(1),                  # Fire for each late element
    ),
    allowed_lateness=Duration(seconds=7200), # Accept up to 2 hours late
    accumulation_mode=AccumulationMode.ACCUMULATING,
)
"""

print("=== Beam Windowing Code Patterns ===")
print(windowing_code)
print("Key concepts:")
print("  - ACCUMULATING: each firing includes ALL data seen so far")
print("  - DISCARDING: each firing includes only NEW data since last firing")
print("  - Watermark: estimate of event-time progress")
print("  - Allowed lateness: how long after watermark to accept late data")

---
## 3. Dataproc Spark Patterns

Show gcloud commands and PySpark code for Dataproc workloads.

In [ ]:
# --- Dataproc Commands ---

dataproc_cmds = """
# Create ephemeral cluster with autoscaling and preemptible workers
gcloud dataproc clusters create analytics-cluster \\
  --region=us-central1 \\
  --num-workers=4 \\
  --worker-machine-type=n2-standard-8 \\
  --num-secondary-workers=8 \\
  --secondary-worker-type=spot \\
  --autoscaling-policy=my-policy \\
  --max-idle=30m \\
  --enable-component-gateway

# Submit PySpark job
gcloud dataproc jobs submit pyspark \\
  --cluster=analytics-cluster \\
  --region=us-central1 \\
  gs://my-bucket/jobs/transform.py \\
  -- --input=gs://raw-data/ --output=gs://processed/

# Dataproc Serverless (no cluster needed)
gcloud dataproc batches submit pyspark \\
  --region=us-central1 \\
  --deps-bucket=gs://my-bucket/deps \\
  gs://my-bucket/jobs/transform.py
"""

print("=== Dataproc gcloud Commands ===")
print(dataproc_cmds)

In [ ]:
# --- Spark vs Beam Comparison ---

comparison = pd.DataFrame({
    'Criteria': ['Streaming', 'Existing Spark code', 'Serverless',
                 'ML in pipeline', 'Hadoop ecosystem', 'Portability',
                 'Exactly-once'],
    'Dataflow (Beam)': [
        'Native, exactly-once, auto-scaling',
        'Requires rewrite to Beam SDK',
        'Always fully serverless',
        'Limited (Beam ML, TFX)',
        'Not applicable',
        'Beam SDK works on multiple runners',
        'Yes (built-in)'
    ],
    'Dataproc (Spark)': [
        'Structured Streaming (at-least-once default)',
        'Run as-is (lift and shift)',
        'Serverless option or managed clusters',
        'Full Spark ML / MLlib ecosystem',
        'Hive, HBase, Presto, Sqoop',
        'Spark runs on Dataproc, EMR, HDInsight',
        'Requires extra effort'
    ]
})

print("=== Spark vs Beam Decision Guide ===")
print(comparison.to_string(index=False))
print()
print("Rule of thumb:")
print("  - New streaming pipeline -> Dataflow")
print("  - Existing Spark/Hadoop code -> Dataproc")
print("  - Need Spark ML -> Dataproc")
print("  - Serverless batch or streaming -> Dataflow")

---
## 4. Cloud Composer DAG Patterns

Show production DAG examples with operators, sensors, and error handling.

In [ ]:
# --- Production DAG Example ---

dag_code = '''
from airflow import DAG
from airflow.utils.dates import days_ago
from airflow.utils.task_group import TaskGroup
from airflow.providers.google.cloud.operators.bigquery import (
    BigQueryInsertJobOperator, BigQueryCheckOperator,
)
from airflow.providers.google.cloud.operators.dataflow import (
    DataflowCreatePythonJobOperator,
)
from airflow.providers.google.cloud.sensors.gcs import GCSObjectExistenceSensor
from datetime import timedelta

default_args = {
    "owner": "data-eng",
    "retries": 3,
    "retry_delay": timedelta(minutes=5),
    "retry_exponential_backoff": True,
    "email_on_failure": True,
    "email": ["alerts@example.com"],
    "sla": timedelta(hours=2),
}

with DAG(
    dag_id="daily_etl_pipeline",
    default_args=default_args,
    schedule_interval="0 6 * * *",  # 6 AM UTC daily
    start_date=days_ago(1),
    catchup=False,
    tags=["production", "etl"],
) as dag:

    # Wait for source file
    wait_for_file = GCSObjectExistenceSensor(
        task_id="wait_for_source",
        bucket="raw-data",
        object="events/{{ ds }}/data.parquet",
        timeout=3600,
        poke_interval=300,
    )

    # Run Dataflow pipeline
    run_dataflow = DataflowCreatePythonJobOperator(
        task_id="run_dataflow",
        py_file="gs://my-bucket/pipelines/transform.py",
        job_name="daily-transform-{{ ds_nodash }}",
        options={
            "input": "gs://raw-data/events/{{ ds }}/",
            "output": "my-project:dataset.events",
        },
    )

    # Data quality checks
    with TaskGroup("quality_checks") as qc_group:
        check_row_count = BigQueryCheckOperator(
            task_id="check_rows",
            sql="SELECT COUNT(*) > 0 FROM `dataset.events` WHERE DATE(ts) = \'{{ ds }}\'",
            use_legacy_sql=False,
        )
        check_nulls = BigQueryCheckOperator(
            task_id="check_nulls",
            sql="SELECT COUNT(*) = 0 FROM `dataset.events` WHERE user_id IS NULL AND DATE(ts) = \'{{ ds }}\'",
            use_legacy_sql=False,
        )

    # Build aggregates
    build_agg = BigQueryInsertJobOperator(
        task_id="build_aggregates",
        configuration={
            "query": {
                "query": "CREATE OR REPLACE TABLE ... AS SELECT ...",
                "useLegacySql": False,
            }
        },
    )

    wait_for_file >> run_dataflow >> qc_group >> build_agg
'''

print("=== Cloud Composer DAG Example ===")
print(dag_code)

---
## 5. Pub/Sub Message Flow Simulation

Simulate Pub/Sub topic/subscription patterns and message routing.

In [ ]:
# --- Simulate Pub/Sub fan-out pattern ---

class PubSubSimulator:
    """Simulates Pub/Sub topic with multiple subscriptions."""
    def __init__(self, topic_name):
        self.topic = topic_name
        self.subscriptions = {}
        self.dead_letter = []
    
    def create_subscription(self, name, filter_fn=None, max_retries=3):
        self.subscriptions[name] = {
            'messages': [], 'filter': filter_fn,
            'max_retries': max_retries, 'processed': 0, 'failed': 0
        }
    
    def publish(self, message):
        for name, sub in self.subscriptions.items():
            if sub['filter'] is None or sub['filter'](message):
                sub['messages'].append(message)
    
    def pull(self, subscription_name, max_messages=10):
        sub = self.subscriptions[subscription_name]
        messages = sub['messages'][:max_messages]
        sub['messages'] = sub['messages'][max_messages:]
        sub['processed'] += len(messages)
        return messages

# Create topic with fan-out
sim = PubSubSimulator('events-topic')
sim.create_subscription('all-events')  # Gets everything
sim.create_subscription('purchases-only', filter_fn=lambda m: m.get('type') == 'purchase')
sim.create_subscription('high-value', filter_fn=lambda m: m.get('amount', 0) > 100)

# Publish messages
messages = [
    {'type': 'purchase', 'user': 'u001', 'amount': 45.99},
    {'type': 'browse', 'user': 'u002', 'amount': 0},
    {'type': 'purchase', 'user': 'u003', 'amount': 250.00},
    {'type': 'purchase', 'user': 'u004', 'amount': 12.50},
    {'type': 'browse', 'user': 'u001', 'amount': 0},
    {'type': 'purchase', 'user': 'u002', 'amount': 500.00},
]

for msg in messages:
    sim.publish(msg)

print("=== Pub/Sub Fan-Out Pattern ===")
print(f"Published {len(messages)} messages to topic '{sim.topic}'")
print()
for name, sub in sim.subscriptions.items():
    pulled = sim.pull(name, max_messages=100)
    print(f"Subscription '{name}': {len(pulled)} messages")
    for msg in pulled:
        print(f"  -> {msg}")
    print()

In [ ]:
# --- Pub/Sub gcloud Commands ---

pubsub_cmds = """
# Create topic with schema enforcement
gcloud pubsub topics create events \\
  --schema=projects/my-project/schemas/event-schema \\
  --message-encoding=JSON

# Create subscription with dead-letter and ordering
gcloud pubsub subscriptions create events-sub \\
  --topic=events \\
  --dead-letter-topic=projects/my-project/topics/events-dlq \\
  --max-delivery-attempts=10 \\
  --enable-message-ordering \\
  --ack-deadline=60

# Direct BigQuery subscription (no pipeline needed!)
gcloud pubsub subscriptions create events-bq-sub \\
  --topic=events \\
  --bigquery-table=my-project:dataset.events \\
  --use-topic-schema \\
  --write-metadata
"""

print("=== Pub/Sub gcloud Commands ===")
print(pubsub_cmds)
print("Key exam points:")
print("  - BigQuery subscription: direct streaming to BQ, no Dataflow needed")
print("  - Dead-letter topic: unprocessable messages after max attempts")
print("  - Message ordering: guaranteed within ordering key, same region")

---
## 6. BigQuery Streaming Insert Comparison

In [ ]:
# --- Streaming Insert Methods Comparison ---

streaming_methods = pd.DataFrame({
    'Method': ['Storage Write API (default)', 'Legacy Streaming API',
              'Pub/Sub BQ Subscription', 'Batch Load'],
    'Latency': ['Seconds', 'Seconds', 'Seconds', 'Minutes'],
    'Cost': ['Free (within quota)', '$0.05/GB', 'Pub/Sub + BQ storage', 'Free'],
    'Guarantee': ['Exactly-once (committed)', 'At-least-once', 'Exactly-once', 'Atomic per job'],
    'Best For': [
        'Default streaming, cost-effective',
        'Legacy pipelines (migrate away)',
        'Simple event streaming, no pipeline code',
        'Large file imports, cost-sensitive'
    ]
})

print("=== BigQuery Streaming Methods ===")
print(streaming_methods.to_string(index=False))
print()
print("Exam tip: Storage Write API is the modern default.")
print("  - Committed streams = exactly-once")
print("  - Pending streams = buffered, commit on demand")
print("  - Default streams = at-least-once (simpler API)")

---
## Summary

| Topic | Key Takeaway |
|-------|-------------|
| **Apache Beam** | Pipeline + PCollection + PTransform. ParDo for element-wise, GroupByKey for aggregation. |
| **Windowing** | Fixed (tumbling), Sliding (overlapping), Session (activity-based), Global (default). |
| **Late Data** | Watermark + allowed_lateness + triggers. ACCUMULATING vs DISCARDING mode. |
| **Dataproc** | Managed Spark/Hadoop. Ephemeral clusters + preemptible VMs. Serverless option available. |
| **Cloud Composer** | Managed Airflow. DAGs with operators, sensors, task groups. Use Composer 2. |
| **Pub/Sub** | At-least-once messaging. Fan-out via multiple subscriptions. BigQuery subscription for direct streaming. |